In [1]:
platformID = 'YT-'

## import libraries

In [2]:
from IPython.display import display

import os
import zipfile

from tqdm import tqdm 
from datetime import datetime

import pandas as pd
pd.set_option('display.max_colwidth', None)

import numpy as np

import re

#import yxdb

import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns 

import psycopg2

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import execute_sql_query
import test_functions

In [4]:
gam_info['lookup_file']

'GAM2025_Lookup.xlsx'

In [5]:
# country
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID', 
                              keep_default_na=False)

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])
week_tester['week_ending'] = pd.to_datetime(week_tester['week_ending'])

# social media accounts
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')

socialmedia_accounts = socialmedia_accounts[(socialmedia_accounts['PlatformID'] == platformID)
                                            & 
                                            (socialmedia_accounts['Status'] == 'active')]
socialmedia_accounts = socialmedia_accounts.rename(columns={'Excluding UK': 'Channel Group'})

channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)
socialmedia_accounts.sample()

,PlatformID,Status,Comment,Channel ID,Channel Name,Service,ServiceID,Channel Group,Channel URL,Channel Username,Linked FB Account,Year
523,YT-,active,NaN,UCsH_ZsmDj75yATm1pwDE2og,Absolutely Fabulous,Studios,WOR,Yes,NaN,NaN,NaN,GAM2025


# Unique Viewers

In [6]:
uniqueViewer_df = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_uniqueViewer.csv")
uniqueViewer_df.sample()


,Channel ID,Channel Name,ServiceID,Channel Group,Channel title,Unique viewers,w/c
4242,UCO_DaPp4ifhMAmiyybSLx9w,BBC Earth Kids,WOR,BBCW Family,BBC Earth Kids,415224.0,2025-03-10


# Country

In [8]:
country_df = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_country_new.csv")
country_df.sample()


,w/c,Channel ID,PlaceID,country_%
441306,2024-08-12,UCUCfrQJCw5GGg7mCp50V-ZQ,PHI,0.002991


# combine UV & country

In [10]:
yt_uv_country = uniqueViewer_df.merge(country_df, 
                            on=['Channel ID', 'w/c'], 
                            how = 'outer', indicator=True)

################################### Testing ################################### 
test_step = 'merging uv & country'

test_functions.test_inner_join(uniqueViewer_df, country_df, ['Channel ID', 'w/c'], '1_YT_18', test_step)

################################### Testing ################################### 

print(yt_uv_country._merge.value_counts())

Inner join test 1_YT_18 failed: Issues found.
Issues with df_left (rows present in df_left but not in df_right)
Issues with df_right (rows present in df_right but not in df_left)
...updating logbook...

_merge
both          946413
right_only      2569
left_only       1529
Name: count, dtype: int64


In [11]:
#yt_uv_country.to_csv('temp_yt_uvCountry.csv', index=None)

In [12]:
yt_uv_country = yt_uv_country[yt_uv_country['_merge'] == 'both'].drop(columns=['_merge'])
yt_uv_country['uv_by_country'] = yt_uv_country['country_%'] * yt_uv_country['Unique viewers']
yt_uv_country.sample()

,Channel ID,Channel Name,ServiceID,Channel Group,Channel title,Unique viewers,w/c,PlaceID,country_%,uv_by_country
138618,UC8zQiuT0m1TELequJ5sp5zw,BBC News - Русская служба,RUS,BBC World Service,BBC News - Русская служба,1484050.0,2024-11-11,PER,0.000021,30.92347


In [13]:
################################### Testing ################################### 
# all weeks 
# all weeks per channel
# all weeks per service
# all weeks per country

# duplicates

# test unique viewer is larger than unique vieewr per country 
# test total of unique vieewr per country == unique viewer
# test country sum == 1

# test allowed values - placeID
# test allowed values - ServiceID

################################### Testing ################################### 

country tests
- [ ] check only one entry per country & week & channel
- [ ] check that sum of unique views per country == unique views gathered from yt clickedicklick
- [ ] check country sum==1 (groupby channel & week == 1)
- [ ] county the number of weeks we have for every channel counntry combination
- [ ] check that no channel name is empty

In [14]:
cols = ['w/c', 'PlaceID', 'ServiceID', 'Channel ID', 'uv_by_country', ]
yt_uv_country[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_uniqueViewer_country.csv", 
                     index=None)


'yt_uv_country.to_csv(f"../data/singlePlatform/input/YouTube/{gam_info[\'file_timeinfo\']}_metric_country.csv", \n                     index=None)'